# CLIP 박스탐색 — 원문제 모범답안2 코드 그대로 (Colab)

**연습 기법** (IOAI 2025 Individual **Pixel** *모범답안2*(개선판, 0.8296→**0.9341**) 와 동일): **CLIP** 으로 이미지의
**가장 정보가 많은 작은 박스**(전체의 ≤6.25%)를 찾는다 — 박스 밖을 검정으로 가린 뒤 **CLIP 분류가 여전히 맞도록**
하는 직사각형을 **coarse-to-fine 탐색**. **이 노트북 코드는 Pixel 모범답안2 골격을 그대로** 따른다:

| Pixel 모범답안2 | 이 노트북 |
|---|---|
| `CLIPModel` (vit-large) | `CLIPModel` (vit-base, 경량) |
| `logits_of(boxes, ni)` (박스 밖 검정→CLIP 코사인 로짓) | 동일 (그대로) |
| coarse-to-fine `best_box` (56×56=6.25%) | 동일 (그대로) |
| target = 전체이미지 CLIP 예측 | 동일 |

**대회**: [Dogs vs Cats](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition) — CLIP 제로샷이 잘 되는 자연이미지
(11번에서 이미 사용). Pixel 처럼 *고양이/개* 텍스트로 박스를 탐색해 **6.25% 픽셀만으로 분류 정확도 유지**가 되는지 본다.

> ⚙️ GPU 권장. 박스탐색은 *라벨 있는 train 샘플*로 시연·평가(이미지당 CLIP 수십 회). 제출용 test 분류는 전체이미지 CLIP
> (12,500장 박스탐색은 8분 한도 초과라 — Pixel 원문제도 698장/8분). ⚠️ API 토큰 평문 — 외부 공유 금지.

## 0. 설치 · 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "transformers", "numpy", "pandas", "pillow", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("준비 완료")

## 1. Kaggle 에서 Dogs vs Cats 다운로드

In [ ]:
import glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("dogs-vs-cats-redux-kernels-edition", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
for fn in ["train.zip", "test.zip"]:
    zp = os.path.join(WORK_DIR, fn)
    if os.path.exists(zp):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
TRAIN_DIR = os.path.join(WORK_DIR, "train"); TEST_DIR = os.path.join(WORK_DIR, "test")
print("train:", len(os.listdir(TRAIN_DIR)), "| test:", len(os.listdir(TEST_DIR)))

## 2. CLIP 로드 & 텍스트 임베딩 (Pixel 모범답안2 그대로)

In [ ]:
##########################
# Your code here
##########################

## 3. logits_of — 박스 밖을 검정으로 가리고 CLIP 로짓 (모범답안2 그대로)
이미지를 224×224 로 정규화(`ni`), 검정 정규화 이미지(`BLACK`)에 박스 영역만 복사해 CLIP vision → 코사인 로짓.

In [ ]:
##########################
# Your code here
##########################

## 4. best_box — coarse-to-fine 박스탐색 (모범답안2 그대로, 56×56=6.25%)
전체이미지 CLIP 예측을 target 으로, 거친 격자에서 최선을 찾고 그 주변을 정밀 탐색해 *target 을 유지하며 margin 최대* 박스 선택.

In [ ]:
##########################
# Your code here
##########################

## 5. 시연·평가 — 전체이미지 vs 6.25% 박스 (라벨 있는 train 샘플)
Pixel 의 목표: *적은 픽셀로 CLIP 정확도 유지*. 전체이미지 CLIP 정확도와, 박스탐색(6.25%) 마스킹 이미지의 정확도를 비교.

In [ ]:
##########################
# Your code here
##########################

## 6. 박스탐색 시각화 (원본 / 박스 / 마스킹)

In [ ]:
##########################
# Your code here
##########################

## 7. 캐글 제출 — test 전체 CLIP 분류 (`id,label`=개 확률)
제출용 test 12,500장은 전체이미지 CLIP 으로 분류(박스탐색은 위 시연이 핵심). dogs-vs-cats 리더보드 형식.

In [ ]:
import pandas as pd
test_files = sorted(glob.glob(os.path.join(TEST_DIR, "*.jpg")), key=lambda p: int(os.path.splitext(os.path.basename(p))[0]))
ids, probs = [int(os.path.splitext(os.path.basename(p))[0]) for p in test_files], []
for i in range(0, len(test_files), 128):
    imgs = [Image.open(p).convert("RGB") for p in test_files[i:i+128]]
    inp = proc(images=imgs, return_tensors="pt").to(DEV)
    with torch.no_grad():
        ie = model.visual_projection(model.vision_model(**inp).pooler_output)
        ie = ie / ie.norm(dim=-1, keepdim=True)
        probs.extend((100*ie @ text_emb.T).softmax(1)[:,1].cpu().numpy().tolist())   # P(dog)
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"id": ids, "label": np.clip(probs, 1e-4, 1-1e-4)}).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| rows:", len(ids))
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [Dogs vs Cats](https://www.kaggle.com/c/dogs-vs-cats-redux-kernels-edition/submit) 에 제출.

Pixel **모범답안2**(coarse-to-fine CLIP 박스탐색) 골격(`logits_of`(박스밖 검정→CLIP)→`best_box`(거친→정밀, 56×56=6.25%))
을 그대로 옮겨, *적은 픽셀로 CLIP 분류 유지* 를 dogs-vs-cats 로 연습. 더: vit-large·격자 촘촘히·정밀단계 확장,
마스킹 이미지로 제출(박스탐색을 test 에도 — 단 시간↑). 11번(전체이미지 CLIP 제로샷)과 비교하면 *박스탐색* 의 효과가 보인다.